## Convolutional Vision Transformer

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [9]:
# NOTE: This is a simplified implementation of the CvT (Convolutional Vision Transformer) module.
# The original paper demonstrates that introducing convolutions in both token embedding and attention
# projection can improve local feature modeling and overall efficiency.

class CvTBlock(nn.Module):
    def __init__(self, in_channels, embed_dim, num_heads, kernel_size=3, stride_q=1, stride_kv=1, mlp_ratio=4, dropout=0.0):
        super(CvTBlock, self).__init__()
        self.num_heads = num_heads
        # NOTE: Convolutional projections for Q, K, and V capture local spatial information better than pure linear projections.
        self.q_conv = nn.Conv2d(in_channels, embed_dim, kernel_size=kernel_size, stride=stride_q,
                                padding=kernel_size//2, bias=False)
        self.k_conv = nn.Conv2d(in_channels, embed_dim, kernel_size=kernel_size, stride=stride_kv,
                                padding=kernel_size//2, bias=False)
        self.v_conv = nn.Conv2d(in_channels, embed_dim, kernel_size=kernel_size, stride=stride_kv,
                                padding=kernel_size//2, bias=False)

        # Linear projection after attention.
        self.proj = nn.Linear(embed_dim, embed_dim)

        # If the spatial dimensions or channels differ, we use a 1x1 convolution to match them.
        if stride_q != 1 or in_channels != embed_dim:
            self.skip_conv = nn.Conv2d(in_channels, embed_dim, kernel_size=1, stride=stride_q)
        else:
            self.skip_conv = None

        # NOTE: BatchNorm2d is used here for normalization, which works naturally with convolutional features.
        self.norm1 = nn.BatchNorm2d(embed_dim)
        self.norm2 = nn.BatchNorm2d(embed_dim)

        # MLP block (feed-forward network) implemented with 1x1 convolutions.
        self.mlp = nn.Sequential(
            nn.Conv2d(embed_dim, int(mlp_ratio * embed_dim), kernel_size=1),
            nn.GELU(),
            nn.Conv2d(int(mlp_ratio * embed_dim), embed_dim, kernel_size=1),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        # x: (B, in_channels, H, W)
        B = x.size(0)
        # Generate Q, K, and V using convolutional projections.
        Q = self.q_conv(x)  # shape: (B, embed_dim, H_q, W_q)
        K = self.k_conv(x)  # shape: (B, embed_dim, H_k, W_k)
        V = self.v_conv(x)  # shape: (B, embed_dim, H_k, W_k)

        B, C, H_q, W_q = Q.shape
        B, C, H_k, W_k = K.shape

        # Flatten the spatial dimensions to create a sequence of tokens.
        Q_flat = Q.view(B, C, -1).transpose(1, 2)  # shape: (B, N_q, C), where N_q = H_q * W_q
        K_flat = K.view(B, C, -1).transpose(1, 2)  # shape: (B, N_k, C)
        V_flat = V.view(B, C, -1).transpose(1, 2)  # shape: (B, N_k, C)

        # Split the channels into multiple heads.
        head_dim = C // self.num_heads
        def reshape_to_heads(x):
            B, N, C = x.shape
            x = x.view(B, N, self.num_heads, head_dim)
            return x.transpose(1, 2)  # shape: (B, num_heads, N, head_dim)

        Q_heads = reshape_to_heads(Q_flat)
        K_heads = reshape_to_heads(K_flat)
        V_heads = reshape_to_heads(V_flat)

        # NOTE: Scaled dot-product attention is computed on these convolutionally projected tokens.
        attn = torch.matmul(Q_heads, K_heads.transpose(-2, -1)) / math.sqrt(head_dim)  # shape: (B, num_heads, N_q, N_k)
        attn = F.softmax(attn, dim=-1)
        attn_out = torch.matmul(attn, V_heads)  # shape: (B, num_heads, N_q, head_dim)

        # Combine attention heads.
        attn_out = attn_out.transpose(1, 2).reshape(B, -1, C)  # shape: (B, N_q, C)

        # Apply linear projection to the combined output.
        attn_out = self.proj(attn_out)  # shape: (B, N_q, C)

        # Reshape the sequence back into the spatial grid.
        attn_out = attn_out.transpose(1, 2).view(B, C, H_q, W_q)

        # Skip connection with possible projection.
        res = self.skip_conv(x) if self.skip_conv is not None else x

        # NOTE: Residual connections facilitate training and enable the model to learn identity mappings.
        out = self.norm1(attn_out + res)

        # Apply the MLP block and add another residual connection.
        out = self.norm2(out + self.mlp(out))
        return out

In [10]:
## CvT Network design
class CvT(nn.Module):
    def __init__(self, in_channels=3, embed_dim=64, num_heads=4, num_layers=2, dropout=0.0):
        super(CvT, self).__init__()
        # NOTE: The convolutional token embedding (here with a large kernel and stride)
        # is key to capturing local patterns before the transformer blocks.
        self.token_embed = nn.Conv2d(in_channels, embed_dim, kernel_size=7, stride=4, padding=3)
        self.bn_embed = nn.BatchNorm2d(embed_dim)

        # Stack multiple CvT blocks.
        self.blocks = nn.Sequential(*[
            CvTBlock(embed_dim, embed_dim, num_heads, kernel_size=3, stride_q=1, stride_kv=1, dropout=dropout)
            for _ in range(num_layers)
        ])

        # Global average pooling and a classifier head for demonstration.
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(embed_dim, 10)  # Example: classify into 10 classes

    def forward(self, x):
        # x: (B, 3, H, W)
        x = self.token_embed(x)  # shape: (B, embed_dim, H/4, W/4)
        x = self.bn_embed(x)
        x = F.gelu(x)
        x = self.blocks(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

In [11]:
# Test the CvT module with a dummy input.
if __name__ == "__main__":
    model = CvT(in_channels=3, embed_dim=64, num_heads=4, num_layers=2, dropout=0.1)
    print("CvT model architecture:\n", model)
    dummy_input = torch.randn(1, 3, 224, 224)  # Example input: 1 image of 224x224 with 3 channels
    output = model(dummy_input)
    print("Output shape:", output.shape)  # Expected output shape: (1, 10)


CvT model architecture:
 CvT(
  (token_embed): Conv2d(3, 64, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3))
  (bn_embed): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (blocks): Sequential(
    (0): CvTBlock(
      (q_conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (k_conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (v_conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (proj): Linear(in_features=64, out_features=64, bias=True)
      (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (norm2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (mlp): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1))
        (1): GELU(approximate='none')
        (2): Conv2d(256, 64, kernel_size=(1, 1), stride=(1, 1))
        (3): Dropout(p=